In [16]:
%%capture
!pip install unsloth
!pip install trl datasets huggingface_hub

In [17]:
from unsloth import FastLanguageModel
import torch

max_seq_length = 2048
dtype = None  # auto-detect
load_in_4bit = True  # QLoRA - fits on free T4

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Llama-3.2-3B-Instruct",
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
)

print("Model loaded successfully")
print(f"Model parameters: {model.num_parameters():,}")

==((====))==  Unsloth 2026.5.5: Fast Llama patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/254 [00:00<?, ?it/s]

Unsloth: Will load unsloth/llama-3.2-3b-instruct-unsloth-bnb-4bit as a legacy tokenizer.


Model loaded successfully
Model parameters: 3,212,749,824


In [18]:
model = FastLanguageModel.get_peft_model(
    model,
    r = 16,                      # LoRA rank — higher = more capacity, more memory
    target_modules = [           # which layers to apply LoRA to
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj"
    ],
    lora_alpha = 16,             # scaling factor, keep equal to r
    lora_dropout = 0,            # 0 is optimal for Unsloth
    bias = "none",               # no bias training
    use_gradient_checkpointing = "unsloth",  # saves memory
    random_state = 42,
)

model.print_trainable_parameters()

trainable params: 24,313,856 || all params: 3,237,063,680 || trainable%: 0.7511


In [19]:
import json
from datasets import Dataset

# Load the preferences.jsonl we built locally
pairs = []
with open("preferences.jsonl", "r", encoding="utf-8") as f:
    for line in f:
        pairs.append(json.loads(line))

print(f"Loaded {len(pairs)} preference pairs")

# Convert to HuggingFace Dataset format
dataset = Dataset.from_list([
    {
        "prompt":   p["prompt"],
        "chosen":   p["chosen"],
        "rejected": p["rejected"],
    }
    for p in pairs
])

print(f"Dataset: {dataset}")
print(f"\nExample prompt:\n{dataset[0]['prompt']}")

Loaded 26 preference pairs
Dataset: Dataset({
    features: ['prompt', 'chosen', 'rejected'],
    num_rows: 26
})

Example prompt:
You are a football expert. Explain concepts clearly with accurate definitions and concrete examples.

Question: What is a high defensive line and what are its risks?


In [20]:
from trl import DPOTrainer, DPOConfig

dpo_config = DPOConfig(
    output_dir = "dpo_output",
    num_train_epochs = 3,          # 3 passes over the 26 pairs
    per_device_train_batch_size = 2,
    gradient_accumulation_steps = 4,  # effective batch size = 8
    learning_rate = 5e-5,
    beta = 0.1,                    # DPO temperature — how strongly to push chosen vs rejected
                                   # 0.1 is standard; lower = more aggressive
    fp16 = not torch.cuda.is_bf16_supported(),
    bf16 = torch.cuda.is_bf16_supported(),
    logging_steps = 5,
    warmup_ratio = 0.1,
    lr_scheduler_type = "cosine",
    save_strategy = "no",          # don't save checkpoints, save at end only
    report_to = "none",            # no wandb
)

trainer = DPOTrainer(
    model = model,
    args = dpo_config,
    train_dataset = dataset,
    processing_class = tokenizer,
)

print("Starting DPO training...")
print(f"  Pairs: {len(dataset)}")
print(f"  Epochs: {dpo_config.num_train_epochs}")
print(f"  Effective batch size: {dpo_config.per_device_train_batch_size * dpo_config.gradient_accumulation_steps}")
print(f"  Beta: {dpo_config.beta}")
print()

trainer_stats = trainer.train()

print(f"\nTraining complete!")
print(f"  Runtime: {trainer_stats.metrics['train_runtime']:.1f}s")
print(f"  Final loss: {trainer_stats.metrics['train_loss']:.4f}")

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


Extracting prompt in train dataset (num_proc=6):   0%|          | 0/26 [00:00<?, ? examples/s]

Applying chat template to train dataset (num_proc=6):   0%|          | 0/26 [00:00<?, ? examples/s]

Tokenizing train dataset (num_proc=6):   0%|          | 0/26 [00:00<?, ? examples/s]

Starting DPO training...
  Pairs: 26
  Epochs: 3
  Effective batch size: 8
  Beta: 0.1



==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 26 | Num Epochs = 3 | Total steps = 12
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 24,313,856 of 3,237,063,680 (0.75% trained)


Unsloth: Will smartly offload gradients to save VRAM!
Unsloth: Double buffering enabled (parallel H2D + compute) for backward pass.


Step,Training Loss,rewards / chosen,rewards / rejected,rewards / accuracies,rewards / margins,logps / chosen,logps / rejected,logits / chosen,logits / rejected
5,0.645960,0.072732,-0.041108,0.470588,0.113840,-324.183716,-430.732635,-0.336453,-0.283834
10,0.314728,0.729667,-0.405604,1.000000,1.135271,-286.842010,-400.982666,-0.345241,-0.273889



Training complete!
  Runtime: 162.5s
  Final loss: 0.4229


In [21]:
# Save the LoRA adapter weights
model.save_pretrained("football-dpo-adapter")
tokenizer.save_pretrained("football-dpo-adapter")

print("Adapter saved to football-dpo-adapter/")

# Also save merged model in 16-bit for GGUF conversion
model.save_pretrained_merged(
    "football-dpo-merged",
    tokenizer,
    save_method = "merged_16bit",
)

print("Merged model saved to football-dpo-merged/")

Adapter saved to football-dpo-adapter/
Found HuggingFace hub cache directory: /root/.cache/huggingface/hub


Fetching 1 files:   0%|          | 0/1 [00:00<?, ?it/s]

Checking cache directory for required files...
Cache check failed: model-00001-of-00002.safetensors not found in local cache.
Not all required files found in cache. Will proceed with downloading.
Checking cache directory for required files...
Cache check failed: tokenizer.model not found in local cache.
Not all required files found in cache. Will proceed with downloading.



Unsloth: Preparing safetensor model files: 100%|██████████| 2/2 [00:00<00:00, 7639.90it/s]


Note: tokenizer.model not found (this is OK for non-SentencePiece models)



Unsloth: Merging weights into 16bit: 100%|██████████| 2/2 [03:46<00:00, 113.27s/it]


Unsloth: Merge process complete. Saved to `/content/football-dpo-merged`
Merged model saved to football-dpo-merged/


In [22]:
# Convert merged model to GGUF format (quantized to 4-bit for Ollama)
model.save_pretrained_gguf(
    "football-dpo-gguf",
    tokenizer,
    quantization_method = "q4_k_m"  # 4-bit quantization, good quality/size tradeoff
)

print("GGUF conversion complete!")

Unsloth: Merging model weights to 16-bit format...
Found HuggingFace hub cache directory: /root/.cache/huggingface/hub


Fetching 1 files:   0%|          | 0/1 [00:00<?, ?it/s]

Checking cache directory for required files...
Cache check failed: model-00001-of-00002.safetensors not found in local cache.
Not all required files found in cache. Will proceed with downloading.
Checking cache directory for required files...
Cache check failed: tokenizer.model not found in local cache.
Not all required files found in cache. Will proceed with downloading.



Unsloth: Preparing safetensor model files: 100%|██████████| 2/2 [00:00<00:00, 14488.10it/s]


Note: tokenizer.model not found (this is OK for non-SentencePiece models)



Unsloth: Merging weights into 16bit: 100%|██████████| 2/2 [03:12<00:00, 96.02s/it]


Unsloth: Merge process complete. Saved to `/content/football-dpo-gguf`
Unsloth: Converting to GGUF format...
==((====))==  Unsloth: Conversion from HF to GGUF information
   \\   /|    [0] Installing llama.cpp might take 3 minutes.
O^O/ \_/ \    [1] Converting HF to GGUF f16 might take 3 minutes.
\        /    [2] Converting GGUF f16 to ['q4_k_m'] might take 10 minutes each.
 "-____-"     In total, you will have to wait at least 16 minutes.

Unsloth: llama.cpp found in the system. Skipping installation.
Unsloth: Preparing converter script...
Unsloth: [1] Converting model into f16 GGUF format.
This might take 3 minutes...
Unsloth: Initial conversion completed! Files: ['football-dpo-gguf_gguf/llama-3.2-3b-instruct.F16.gguf']
Unsloth: [2] Converting GGUF f16 into q4_k_m. This might take 10 minutes...
Unsloth: Model files cleanup...
Unsloth: All GGUF conversions completed successfully!
Generated files: ['football-dpo-gguf_gguf/llama-3.2-3b-instruct.Q4_K_M.gguf']
Unsloth: example usage for 

In [23]:
from google.colab import files

# Download the GGUF model file (~2GB - will take a few minutes)
files.download("football-dpo-gguf_gguf/llama-3.2-3b-instruct.Q4_K_M.gguf")

# Download the Modelfile (tiny text file)
files.download("football-dpo-gguf_gguf/Modelfile")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>